In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sharathraoram/datatheseisqwen/qwen_vague_final.csv
/kaggle/input/datasets/sharathraoram/datatheseisqwen/prompt_bank_final.csv
/kaggle/input/datasets/sharathraoram/datatheseisqwen/qwen_original_final.csv


In [2]:
!pip -q install spacy tqdm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 78.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import pandas as pd
import numpy as np
import re
import spacy

from tqdm.auto import tqdm

tqdm.pandas()

nlp = spacy.load("en_core_web_sm")

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/sharathraoram/datatheseisqwen/qwen_vague_final.csv
/kaggle/input/datasets/sharathraoram/datatheseisqwen/prompt_bank_final.csv
/kaggle/input/datasets/sharathraoram/datatheseisqwen/qwen_original_final.csv


In [5]:
INPUT_FILE = "/kaggle/input/datasets/sharathraoram/datatheseisqwen/qwen_original_final.csv"

OUTPUT_FILE = "qwen_original_clarity_specificity.csv"

In [6]:
HEDGE_WORDS = {

    "maybe","perhaps","possibly","might","could","may",

    "seems","appears","apparently","generally",

    "approximately","about","around",

    "kind","sort","etc",

    "something","someone","somewhere",

    "likely","probably"

}

In [7]:
def clarity_specificity(text):

    if pd.isna(text):

        return pd.Series([0,0,0,0])

    text = str(text)

    words = re.findall(r"\b\w+\b", text.lower())

    total_words = len(words)

    if total_words == 0:

        return pd.Series([0,0,0,0])

    hedge_count = sum(word in HEDGE_WORDS for word in words)

    hedge_ratio = hedge_count / total_words

    doc = nlp(text)

    specific_count = 0

    for token in doc:

        if token.like_num:

            specific_count += 1

    specific_count += len(doc.ents)

    specificity_index = specific_count / total_words

    return pd.Series(

        [

            hedge_count,

            hedge_ratio,

            specific_count,

            specificity_index

        ]

    )

In [8]:
df = pd.read_csv(INPUT_FILE)

df[
    [
        "hedge_word_count",
        "hedge_ratio",
        "specific_term_count",
        "specificity_index"
    ]
] = df["response"].progress_apply(clarity_specificity)

  0%|          | 0/1024 [00:00<?, ?it/s]

In [9]:
df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved Successfully:", OUTPUT_FILE)

display(
    df[
        [
            "hedge_word_count",
            "hedge_ratio",
            "specific_term_count",
            "specificity_index"
        ]
    ].head(20)
)

Saved Successfully: qwen_original_clarity_specificity.csv


,hedge_word_count,hedge_ratio,specific_term_count,specificity_index
0,0.0,0.000000,37.0,0.521127
1,0.0,0.000000,6.0,0.044444
2,0.0,0.000000,0.0,0.000000
3,0.0,0.000000,20.0,0.112994
4,0.0,0.000000,27.0,0.139175
5,0.0,0.000000,27.0,0.177632
6,1.0,0.006329,19.0,0.120253
7,0.0,0.000000,6.0,0.040268
8,6.0,0.031915,10.0,0.053191
9,0.0,0.000000,6.0,0.030151
